# Pendulum Property-Driven Training

This notebook reproduces the property-driven learning loop from the DICE tutorial,
but applied to the *pendulum controller* network used in our repository.

We will:
1. Load a small pendulum dataset (`theta`, `theta_dot`, `u_ref`).
2. Train an MLP controller with a *saturation property loss*  
   enforcing that for all valid states, `u_min <= u <= u_max`.
3. Export the model to ONNX.
4. Verify the trained controller with **Vehicle** + **Marabou**.


### Setup

In [1]:
import torch
import torch.nn as nn
# from property_driven_ml.training import PropertyDrivenTrainer
# from property_driven_ml.constraints import Constraint
# from property_driven_ml.constraints.postconditions import Postcondition
# from property_driven_ml.logics import Logic

from pendulum_ml.models.registry import make_model


if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Using device:", device)


: 

### Load Data and Normalisation

In [ ]:
# Paths from your repo
data_dir = Path("../data/processed/pendulum")

X = torch.load(data_dir / "X.pt")  # [N, 3]: theta, theta_dot, error
y = torch.load(data_dir / "y.pt")  # [N, 1]: torque target

with open(data_dir / "norms.json") as f:
    norms = json.load(f)

mu = torch.tensor(norms["mu"][0], dtype=torch.float32)
sd = torch.tensor(norms["sd"][0], dtype=torch.float32)

# Move data and normalization stats to the selected device so training doesn't hit
# "Tensor is on cpu but expected on mps" errors when model/device differ.
X = X.to(device)
y = y.to(device)
mu = mu.to(device)
sd = sd.to(device)

print(f"Dataset: {X.shape[0]} samples, mean/std loaded and moved to {device}.")


### Define the model

In [ ]:
def load_snapshot(run: str):
    snap = Path("../experiments") / run / "config.json"
    if not snap.exists():
        raise SystemExit(f"Missing snapshot: {snap}")
    return json.loads(snap.read_text())

run = "run-20251107-173544"
cfg = load_snapshot(run)
model_name = cfg["model"]["name"]
kwargs = cfg["model"].get(model_name, {})
model = make_model(model_name, **kwargs).to(device)
print(model)


### Define the Property Loss (Saturation)

In [ ]:
# Hyperparameters for property
u_min, u_max = -10.0, 10.0

def saturation_loss(u):
    """ Penalise outputs outside [u_min, u_max]. """
    lower = torch.relu(u_min - u)
    upper = torch.relu(u - u_max)
    return (lower + upper).mean()


### Training Loop with Property-Driven Loss

In [ ]:
opt = optim.Adam(model.parameters(), lr=1e-3)
n_epochs = 250
lambda_prop = 15.0  # weight for property term

train_loss, prop_violation = [], []

for epoch in range(n_epochs):
    model.train()
    opt.zero_grad()

    # Normalise inputs
    Xn = (X - mu) / sd
    u_pred = model(Xn).squeeze(1)
    u_true = y.squeeze(1)

    # Supervised loss (teacher torque)
    mse = nn.MSELoss()(u_pred, u_true)

    # Property loss
    L_prop = saturation_loss(u_pred)

    loss = mse + lambda_prop * L_prop
    loss.backward()
    opt.step()

    train_loss.append(loss.item())
    prop_violation.append(L_prop.item())

    if epoch % 5 == 0:
        print(f"Epoch {epoch:03d}: total={loss.item():.4f}  mse={mse.item():.4f}  prop={L_prop.item():.4f}")


### Visualise Training Dynamics

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(train_loss, label="Total loss")
plt.plot(prop_violation, label="Property penalty")
plt.xlabel("Epoch")
plt.legend()
plt.title("Training with Saturation Property")
plt.show()


### Check Property Satisfaction

In [ ]:
N_test = 5000
theta = torch.empty(N_test).uniform_(-3.1416, 3.1416)
theta_dot = torch.empty(N_test).uniform_(-10, 10)
err = -theta  # assume setpoint=0

Xtest = torch.stack([theta, theta_dot, err], dim=1).to(device)
Xtest_n = (Xtest - mu) / sd
u_pred = model(Xtest_n).detach().cpu().squeeze()

violations = ((u_pred < u_min) | (u_pred > u_max)).float().mean()
print(f"Fraction of violations for property driven training: {violations*100:.2f}%")


# Load ckpt for comparison
ckpt_path = Path("../models/checkpoints") / f"{run}.pt"

model_ckpt = make_model(model_name, **kwargs).to(device)
model_ckpt.load_state_dict(torch.load(ckpt_path, map_location=device))
model_ckpt.eval()
u_pred_ckpt = model_ckpt(Xtest_n).detach().cpu().squeeze()
violations_ckpt = ((u_pred_ckpt < u_min) | (u_pred_ckpt > u_max)).float().mean()
print(f"Fraction of violations for checkpoint model: {violations_ckpt*100:.2f}%")


### Export to ONNX

In [ ]:
out_path = Path("../models/onnx/pendulum_ctrl_property.onnx")
out_path.parent.mkdir(parents=True, exist_ok=True)

dummy = torch.zeros(1, 3, dtype=torch.float32).to(device)
torch.onnx.export(model, dummy, out_path,
                  input_names=["input"], output_names=["output"],
                  opset_version=13)
print("Exported:", out_path)

In [ ]:
!vehicle verify \
    --specification ../specs/pendulum_controller.vcl \
    --network ctrlNet:../models/onnx/pendulum_ctrl_property.onnx \
    $(cat ../specs/pendulum_params.cli) \
    --verifier Marabou